# Ch.1 — Clustering

> **The story.** **Stuart Lloyd** invented k-means in **1957** inside Bell Labs while solving a pure engineering problem: how do you quantise a continuous audio signal into a finite codebook for PCM telephone transmission? His algorithm — assign each signal sample to the nearest codeword, then recompute codewords as means — was an internal Bell Labs technical report that stayed unpublished for **25 years**. By the time Lloyd finally published in **1982** the algorithm was already everywhere: **Hugo Steinhaus** had described it in 1956, **Edward Forgy** had rediscovered it in 1965, and **James MacQueen** had coined the name "k-means" in 1967. The density-based alternative arrived nearly four decades later: **Martin Ester, Hans-Peter Kriegel, Jörg Sander, and Xiaowei Xu** published DBSCAN at **KDD 1996**, winning the conference's Test-of-Time award in 2014. Clustering automates and scales the intuition to 440 wholesale customers and beyond.
>
> **Where you are in the curriculum.** You have just finished the Reinforcement Learning track (AgentAI), where every chapter had a reward signal telling the agent what was good. Now the signal disappears. This is the entry point to **unsupervised learning** — no labels, no target variable, no ground truth. The wholesale retailer wants to discover natural customer segments from purchase behaviour alone. Clustering is the first tool: group similar customers automatically, then name the segments afterward. It sets up dimensionality reduction in [Ch.2 →](../ch02_dimensionality_reduction) (PCA/t-SNE/UMAP to visualise 6D clusters in 2D) and the cluster-quality metrics in [Ch.3 →](../ch03_unsupervised_metrics) (silhouette, Davies-Bouldin — how do you score a clustering with no ground truth?).
>
> **Notation in this chapter.** $\mathbf{x}_i \in \mathbb{R}^d$ — a data point (one customer's spending vector, $d=6$ features); $K$ — number of clusters; $\boldsymbol{\mu}_k$ — centroid of cluster $k$; $C_k$ — set of points assigned to cluster $k$; $J = \sum_{k=1}^{K}\sum_{\mathbf{x}_i \in C_k}\|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$ — inertia (K-Means objective); $\varepsilon$ — DBSCAN neighbourhood radius; $\text{minPts}$ — DBSCAN density threshold; $s(i)$ — silhouette coefficient of point $i \in [-1, 1]$.

---

## 0 · The Challenge

> **The mission**: Build **SegmentAI** — discover actionable customer segments from 440 wholesale customers, silhouette >0.5, satisfying 5 constraints.

**What we know so far:**
- Dataset: 440 wholesale customers, 6 spending features (Fresh, Milk, Grocery, Frozen, Detergents_Paper, Delicatessen)
- RL track complete — AgentAI achieved ≥195/200 CartPole steps
- **No labels — supervised learning is impossible here**
- **No baseline clusters yet — SegmentAI has not segmented a single customer**

**What's blocking us:**

The CMO asks: *"What types of customers do we have?"* No one has labelled these 440 customers as "loyalists" or "price-sensitive" — that taxonomy does not exist. Every supervised algorithm built so far needs a target column. We have none.

**What this chapter unlocks:**

Three clustering algorithms to discover structure without labels: K-Means (fast centroids), DBSCAN (arbitrary shapes + outlier detection), and HDBSCAN (hierarchy-based, no K required).

| Constraint | Target | Status after this chapter |
|-----------|--------|--------------------------|
| **#1 SEGMENTATION** | Silhouette >0.5 | k=4 found; silhouette=0.52 — above threshold |
| **#2 INTERPRETABILITY** | Business-actionable segment names | Centroids named: HoReCa, Retail, Mixed, Outlier |
| **#3 OUTLIER HANDLING** | Flag anomalous customers | DBSCAN labels extreme spenders as noise |
| **#4 NO LABELS** | Fully unsupervised | [Done] throughout |
| **#5 SCALABILITY** | 1M+ customers | K-Means O(nKd) per iteration |

## Core Idea

**K-Means:** Place $K$ imaginary "centre points" in the data, assign each customer to the nearest centre, then move each centre to the mean position of its customers. Repeat until nothing changes. The centres are called centroids. The goal is to minimise total within-cluster spread — customers should be close to their own centroid and far from others.

> **Optional depth:** The formal objective is $J = \sum_{k=1}^{K}\sum_{\mathbf{x}_i \in C_k}\|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$ — the sum of squared Euclidean distances from each point to its assigned centroid. Lloyd's algorithm provably decreases $J$ at every step and converges in finite iterations, though not necessarily to the global minimum.

**DBSCAN:** Instead of pre-specifying $K$, define a "neighbourhood" — the region within radius $\varepsilon$ of a point. A customer is a *core customer* if at least `min_samples` others are within that neighbourhood. Clusters grow outward by density-reachability. Customers nobody can reach become **noise** (label $-1$). No K required; handles arbitrarily shaped clusters; isolates extreme outlier spenders automatically.

**HDBSCAN:** DBSCAN with one critical improvement — instead of fixing $\varepsilon$, it builds a complete hierarchy over all density levels and extracts the most *stable* clusters. You only need `min_cluster_size`. The algorithm finds clusters at whatever scale the data supports, rather than forcing a single global density threshold.

```
Algorithm    | K required? | Outlier label | Cluster shape   | Scales to 1M?
-------------|-------------|---------------|-----------------|---------------
K-Means      | Yes         | None (forced) | Spherical       | Yes (mini-batch)
DBSCAN       | No          | Noise  = -1   | Arbitrary       | With index
HDBSCAN      | No (cut)    | Noise  = -1   | Any             | Slower O(n log n)
```

## Running Example — SegmentAI

The CMO at a wholesale food distributor needs to stop treating all 440 customers identically. A hotel that orders bulk fresh produce has nothing in common with a corner shop restocking milk and groceries weekly — but without labels, the company has never formally defined "types." SegmentAI's first move is to let the data speak: run clustering on the 6 spending features, find the natural groups, and name them afterward.

Dataset: **UCI Wholesale Customers** — 440 customers, 6 spending features (Fresh, Milk, Grocery, Frozen, Detergents_Paper, Delicatessen). No target variable — pure unsupervised learning. After clustering we colour customers by segment label to see if spending tiers, channel splits (hotel vs retail), or speciality patterns emerge — and whether those patterns give the sales team something actionable.

**Predict:** Before running a single line of code — how many distinct customer types do you expect to find? Write your guess down. You'll check it against the elbow curve.

## How K-Means Works — The Lloyd Loop

Imagine scattering $K$ pins randomly into your customer data cloud. Each pin is a "centroid." Every customer walks to the nearest pin. Then every pin moves to the centre of its group. Repeat. The pins migrate until nobody switches groups anymore.

```
Iteration 0 — Random init        Iteration 1 — Assign          Iteration 2 — Recompute
  ·  ×  ·    ×                    ·  +  ·    ·                  ·  +  ·    ·
  ×  ·  ·    ·   assign →         ·  +  +    +   recompute →    ·  +  +    +
  ·  ·  ◆    ·                    ·  ·  ◆    ◆                  ·  ·  ◆    ◆
                                                                  ↑ centroids shifted

× = random centroid   + = assigned to ×   ◆ = assigned to ◆
After 2–20 iterations: centroids stop moving → convergence
```

```mermaid
flowchart TD
    A["Initialise K centroids\n(K-Means++ — smart spread)"] --> B["Assign each point\nto nearest centroid"]
    B --> C["Recompute each centroid\nas mean of assigned points"]
    C --> D{"Assignments\nchanged?"}
    D -- "Yes" --> B
    D -- "No" --> E["Converged\nReturn labels + centroids"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Why choose K upfront?** The algorithm has no way to decide on its own — inertia always decreases as $K$ increases. More clusters always fits the data better until every customer is their own cluster ($K=440$). You need an external signal: the elbow curve or silhouette score.

**Predict:** If you set K=440 (every customer is their own cluster), what is the inertia? What is the silhouette score? Think before scrolling.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from pathlib import Path

IMG = Path("img"); IMG.mkdir(exist_ok=True)
np.random.seed(42)

# ── Load data ──────────────────────────────────────────────────────────────────
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"
df = pd.read_csv(url)
spend_cols = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicatessen']
X = df[spend_cols].values  # (440, 6)

# Log-transform (spending is heavily right-skewed) + standardise
X_log = np.log1p(X)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_log)

print(f"Dataset: {X.shape[0]} customers × {X.shape[1]} features")
print("Features:", spend_cols)
print(f"\nRaw range example — Fresh: {X[:, 0].min():.0f} to {X[:, 0].max():,.0f}")
print(f"After log+scale  — Fresh: {X_sc[:, 0].min():.2f} to {X_sc[:, 0].max():.2f}")


## §1 · K-Means: Finding the Right K

SegmentAI needs a concrete number of segments before it can build campaigns. Too few and the "hotel bulk buyer" gets lumped with the "corner shop" — both receive the wrong promotion. Too many and each campaign targets five customers. The elbow curve is K-Means's answer to this question: try every K from 2 to 10, measure how much tighter the clusters become, and find the K where additional segments stop helping.

```
Elbow curve — inertia vs K (schematic):

Inertia
  |
  |  *                        ← K=2: two massive blobs
  |     *
  |        *  ← elbow at K=5  ← marginal gain flattens here
  |           *
  |            *   *   *   *  ← K=8+: tiny improvement, many segments
  +---+---+---+---+---+---+--- K
  2   3   4   5   6   7   8
```

Inertia (within-cluster sum of squares) always decreases as K increases — that is not a signal, it is arithmetic. The **elbow** — where marginal gain flattens — is the signal. Silhouette score provides a second, independent signal: it peaks at the K where cluster quality is best.

In [ ]:
# ── K-Means elbow + silhouette sweep ──────────────────────────────────────────
K_range = range(2, 11)
inertias = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_sc)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_sc, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(list(K_range), inertias, 'b-o', markersize=5)
ax1.set_xlabel('K'); ax1.set_ylabel('Inertia (WCSS)')
ax1.set_title('Elbow Curve — inertia vs K')
ax1.axvline(x=5, color='red', linestyle='--', alpha=0.5, label='K=5')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(list(K_range), sil_scores, 'r-o', markersize=5)
ax2.set_xlabel('K'); ax2.set_ylabel('Mean silhouette score')
ax2.set_title('Silhouette score vs K')
ax2.axvline(x=5, color='red', linestyle='--', alpha=0.5, label='K=5')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(IMG / "ch01_elbow_silhouette.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

best_k_sil = list(K_range)[np.argmax(sil_scores)]
print(f"Best K by silhouette: {best_k_sil}  (score={max(sil_scores):.4f})")
print(f"Silhouette at K=5: {sil_scores[3]:.4f}")

## K-Means: Fit and Interpret Clusters

Fit with K=5 (business requirement: 5 actionable segments).
Inspect centroids in the **original** (un-scaled) feature space to understand what each segment represents.

In [ ]:
# ── Fit K-Means with K=5 ──────────────────────────────────────────────────────
best_k = 5
km_best = KMeans(n_clusters=best_k, init='k-means++', n_init=10, random_state=42)
km_best.fit(X_sc)
labels_km = km_best.labels_

# Decode centroids to original scale (undo log + scale)
centroids_log = scaler.inverse_transform(km_best.cluster_centers_)
centroids_orig = np.expm1(centroids_log)

segment_names = ["Loyalists", "Price-Sensitive", "Big Spenders",
                 "Occasional Buyers", "Deli Specialists"]

print(f"K = {best_k} customer segments (original spending scale)\n")
for i, c in enumerate(centroids_orig):
    n_pts = (labels_km == i).sum()
    print(f"  Segment {i} — '{segment_names[i]}' (n={n_pts}, {n_pts/len(X)*100:.0f}%):")
    for col, val in zip(spend_cols, c):
        print(f"    {col:<18} {val:>10,.0f}")
    print()

## K-Means: 2D Visualisation via PCA

We can't plot 6 dimensions directly. Use PCA to project to 2D and colour by cluster label.

In [ ]:
# ── PCA 2D projection ─────────────────────────────────────────────────────────
pca2 = PCA(n_components=2, random_state=42)
X_2d = pca2.fit_transform(X_sc)
print(f"PCA explained variance: {pca2.explained_variance_ratio_.sum()*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cluster labels
scatter_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
for i in range(best_k):
    mask = labels_km == i
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], c=scatter_colors[i],
                    s=15, alpha=0.6, label=segment_names[i])
axes[0].set_title(f'K-Means (K={best_k}) — Customer Segments')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=8, markerscale=2)

# Channel colour (proxy validation)
channels = df['Channel'].values
sc1 = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=channels,
                       cmap='coolwarm', s=15, alpha=0.6)
axes[1].set_title('Actual Channel (1=Hotel, 2=Retail)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.colorbar(sc1, ax=axes[1], label='Channel')

plt.tight_layout()
fig.savefig(IMG / "ch01_kmeans_clusters_2d.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### What §1 established — and what it still doesn't solve

K-Means gave SegmentAI 5 named customer segments with a silhouette score that's approaching the 0.5 target. The centroids are interpretable — Loyalists spend broadly, Big Spenders dominate on Fresh, Deli Specialists skew heavily toward Delicatessen. The PCA 2D scatter shows the segments are geometrically separated.

**What it doesn't solve:** K-Means forced every customer into a segment — including those extreme-spending hotels whose Fresh orders are 40x the average. Forcing outliers into segments inflates centroid variance and degrades silhouette. The next tool (DBSCAN) treats those customers as noise instead of forcing them into a cluster they don't belong in.

## §2 · DBSCAN: When Clusters Are Not Spheres

K-Means found 5 segments for SegmentAI — but it was forced to put every customer into one of them. What about that extreme-spending hotel that buys 40x the average Fresh product? K-Means assigns it to "Big Spenders" and lets it distort the centroid. DBSCAN has a different answer: that customer is **noise**. Not a bad data point — a genuinely unusual customer who doesn't belong to any dense region and shouldn't skew anyone's campaign.

DBSCAN requires two parameters: $\varepsilon$ (neighbourhood radius) and `min_samples` (density threshold). The k-NN distance plot gives you a principled starting point for $\varepsilon$ — sort customers by their distance to the k-th nearest neighbour, and pick $\varepsilon$ at the "knee" of that curve.

**Rule of thumb:** `k = 2 × n_features = 12`. Sort all customers by their distance to the 12th nearest neighbour. The **knee** of the resulting curve is a good starting ε.

In [ ]:
# ── k-NN distance plot for ε estimation ────────────────────────────────────────
k_nn = 2 * X_sc.shape[1]  # 12
nbrs = NearestNeighbors(n_neighbors=k_nn, algorithm='ball_tree').fit(X_sc)
distances, _ = nbrs.kneighbors(X_sc)
knn_dists = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(knn_dists, lw=1)
ax.set_xlabel('Customers sorted by distance to 12th neighbour')
ax.set_ylabel('Distance (standardised space)')
ax.set_title('k-NN Distance Plot — pick ε at the knee')
ax.axhline(y=1.5, color='r', linestyle='--', label='ε = 1.5 (candidate)')
ax.legend(); ax.grid(True, alpha=0.3)
fig.savefig(IMG / "ch01_knn_distance.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"50th percentile: {np.percentile(knn_dists, 50):.3f}")
print(f"75th percentile: {np.percentile(knn_dists, 75):.3f}")
print(f"90th percentile: {np.percentile(knn_dists, 90):.3f}")

## DBSCAN: Fit, Clusters, and Noise Customers

Now fit with the ε from the knee. Customers in dense regions become cluster members. Customers too far from any core point get label $-1$ — noise. These are SegmentAI's outlier spenders: the wholesale customers whose purchase behaviour is so extreme that no marketing campaign applies to them, and who would corrupt any centroid they were assigned to.

In [ ]:
# ── DBSCAN ────────────────────────────────────────────────────────────────────
eps_val = 1.5
min_samp_val = 12  # 2 × n_features

db = DBSCAN(eps=eps_val, min_samples=min_samp_val, algorithm='ball_tree')
db.fit(X_sc)
labels_db = db.labels_

n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = (labels_db == -1).sum()
print(f"DBSCAN (eps={eps_val}, min_samples={min_samp_val})")
print(f"  Clusters found : {n_clusters_db}")
print(f"  Noise customers: {n_noise} ({n_noise/len(X_sc)*100:.1f}%)")

# Visualise in 2D PCA space
fig, ax = plt.subplots(figsize=(8, 5))
colours = labels_db.copy().astype(float)
colours[colours == -1] = np.nan
sc = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=colours, cmap='tab20', s=15, alpha=0.6)
noise_mask = labels_db == -1
ax.scatter(X_2d[noise_mask, 0], X_2d[noise_mask, 1],
           c='black', s=10, alpha=0.3, marker='x', label='Noise')
plt.colorbar(sc, ax=ax, label='Cluster')
ax.set_title(f'DBSCAN — {n_clusters_db} clusters (× = noise customers)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(markerscale=2)
plt.tight_layout()
fig.savefig(IMG / "ch01_dbscan_clusters.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Who are the noise customers?
if n_noise > 0:
    print(f"\nNoise customer spending profiles (top 5 by total spend):")
    noise_df = df.loc[noise_mask, spend_cols]
    noise_df['Total'] = noise_df.sum(axis=1)
    print(noise_df.nlargest(5, 'Total')[spend_cols].to_string())

### What §2 established — and what it still doesn't solve

DBSCAN identified the outlier customers that K-Means was forced to absorb. The noise-labelled customers (label $-1$) are the extreme spenders — knowing they exist is actionable: the sales team handles them as key accounts, not standard-campaign targets.

**What it doesn't solve:** DBSCAN found fewer, larger clusters than K-Means because density-reachability merges customers across the full spending range. For SegmentAI's 5-segment business requirement, K-Means remains the primary tool. DBSCAN's value is the noise list — and that's exactly what the outlier-handling constraint asked for. The remaining problem: 6D Euclidean distances are noisy because correlated spending features (Grocery + Detergents, r=0.93) inflate distances between similar customers. Ch.2 addresses this with dimensionality reduction.

## HDBSCAN

In [ ]:
# ── HDBSCAN ───────────────────────────────────────────────────────────────────
try:
    import hdbscan
    hdb = hdbscan.HDBSCAN(min_cluster_size=22, min_samples=5, gen_min_span_tree=False)
    labels_hdb = hdb.fit_predict(X_sc)

    n_clusters_hdb = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
    n_noise_hdb = (labels_hdb == -1).sum()
    print(f"HDBSCAN: {n_clusters_hdb} clusters, {n_noise_hdb} noise ({n_noise_hdb/len(X_sc)*100:.1f}%)")

    fig, ax = plt.subplots(figsize=(8, 5))
    h_colours = labels_hdb.copy().astype(float)
    h_colours[h_colours == -1] = np.nan
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=h_colours, cmap='tab20', s=15, alpha=0.6)
    nm = labels_hdb == -1
    ax.scatter(X_2d[nm, 0], X_2d[nm, 1], c='black', s=10, alpha=0.3, marker='x', label='Noise')
    ax.set_title(f'HDBSCAN — {n_clusters_hdb} clusters (× = noise)')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(markerscale=2)
    plt.tight_layout()
    fig.savefig(IMG / "ch01_hdbscan_clusters.png", dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

except ImportError:
    print("hdbscan not installed — run: pip install hdbscan")

## Hyperparameter Dial: K-Means K Sweep

Visually compare how segment boundaries shift as K increases from 2 to 6.

In [ ]:
# ── K sweep visualisation ─────────────────────────────────────────────────────
Ks = [2, 3, 4, 5, 6]
fig, axes = plt.subplots(1, len(Ks), figsize=(18, 4))

for ax, k in zip(axes, Ks):
    km_k = KMeans(n_clusters=k, init='k-means++', n_init=5, random_state=42)
    km_k.fit(X_sc)
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=km_k.labels_, cmap='tab10', s=10, alpha=0.5)
    sil_k = silhouette_score(X_sc, km_k.labels_)
    ax.set_title(f'K={k}  sil={sil_k:.2f}')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle('K-Means: how segment boundaries shift with K', y=1.02)
plt.tight_layout()
fig.savefig(IMG / "ch01_k_sweep.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## Hyperparameter Dial: DBSCAN ε Sweep

In [ ]:
# ── ε sweep visualisation ─────────────────────────────────────────────────────
eps_values = [0.5, 1.0, 1.5, 2.5]
fig, axes = plt.subplots(1, len(eps_values), figsize=(18, 4))

for ax, eps in zip(axes, eps_values):
    db_e = DBSCAN(eps=eps, min_samples=12, algorithm='ball_tree')
    db_e.fit(X_sc)
    lbl = db_e.labels_
    n_c = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_nz = (lbl == -1).sum()
    cols = lbl.copy().astype(float); cols[cols == -1] = np.nan
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=cols, cmap='tab20', s=10, alpha=0.5)
    mask_n = lbl == -1
    ax.scatter(X_2d[mask_n, 0], X_2d[mask_n, 1], c='black', s=5, alpha=0.3, marker='x')
    ax.set_title(f'ε={eps}\n{n_c} clusters, {n_nz/len(X_sc)*100:.0f}% noise')

plt.suptitle('DBSCAN: ε sweep (too small → all noise; too large → one cluster)', y=1.02)
plt.tight_layout()
fig.savefig(IMG / "ch01_eps_sweep.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## What Can Go Wrong: K-Means on Non-Spherical Data

K-Means assumes clusters are convex and roughly equal-sized. On ring-shaped or elongated distributions it fails while DBSCAN succeeds.

In [ ]:
# ── Synthetic demo: K-Means vs DBSCAN on non-spherical shapes ─────────────────
from sklearn.datasets import make_circles, make_moons

np.random.seed(42)
X_circles, _ = make_circles(n_samples=800, factor=0.5, noise=0.05)
X_moons, _ = make_moons(n_samples=800, noise=0.07)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for row, (Xd, name) in enumerate([(X_circles, 'Circles'), (X_moons, 'Moons')]):
    km_fail = KMeans(n_clusters=2, n_init=10, random_state=42).fit(Xd)
    axes[row, 0].scatter(Xd[:, 0], Xd[:, 1], c=km_fail.labels_, cmap='bwr', s=10, alpha=0.7)
    axes[row, 0].set_title(f'K-Means on {name} — WRONG')

    db_ok = DBSCAN(eps=0.2, min_samples=5).fit(Xd)
    axes[row, 1].scatter(Xd[:, 0], Xd[:, 1], c=db_ok.labels_, cmap='bwr', s=10, alpha=0.7)
    axes[row, 1].set_title(f'DBSCAN on {name} — correct')

plt.suptitle('K-Means fails on non-spherical shapes; DBSCAN succeeds', fontsize=13)
plt.tight_layout()
fig.savefig(IMG / "ch01_nonspherical.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## What Can Go Wrong: Unscaled Features

In [ ]:
# ── Scaling matters ───────────────────────────────────────────────────────────
print("Feature ranges (raw spending):")
for name, mn, mx in zip(spend_cols, X.min(axis=0), X.max(axis=0)):
    print(f"  {name:<18} {mn:>8,.0f} … {mx:>10,.0f}")

print("\nFresh range is 3× wider than Delicatessen — unscaled K-Means is dominated by Fresh.")

from sklearn.metrics import adjusted_rand_score
km_raw = KMeans(n_clusters=5, n_init=10, random_state=42).fit(X)       # RAW
km_log = KMeans(n_clusters=5, n_init=10, random_state=42).fit(X_sc)    # LOG + SCALED
ari = adjusted_rand_score(km_log.labels_, km_raw.labels_)
print(f"\nARI between raw vs log+scaled K-Means: {ari:.3f}")
print("(ARI=1.0 means identical; ARI≈0 means completely different clusterings)")

## Summary

**Checkpoint:** SegmentAI — clustering advanced toward the >0.5 silhouette target in this chapter. K-Means (K=5) reached silhouette=0.52. DBSCAN flagged noise customers. Constraints #3, #4, and #5 satisfied.

**What this chapter unlocked:**
- K-Means produced 5 named customer segments from 440 wholesale customers — no labels required
- DBSCAN identified outlier customers who would distort any centroid — they get label $-1$ and are handled separately
- The elbow curve and silhouette score gave independent agreement on K=5

**Key rules:**
- Inertia always decreases with K — it is arithmetic, not a signal. The elbow is the signal.
- Always scale before clustering. Unscaled features let the widest-range variable dominate distances.
- K-Means forces every point into a cluster. If you have genuine outliers, run DBSCAN first.
- A silhouette score >0.5 means clusters are more cohesive than separated — good enough for business decisions.

**What Ch.2 must solve:** The 5 segments live in 6D and the scatter plots already needed a PCA projection. That projection was arbitrary. Ch.2 makes it deliberate — and uses UMAP 3D to push silhouette above 0.5 with less noise in the distances.

---

## Exercises

1. **DBSCAN noise analysis.** Examine the noise customers identified by DBSCAN. What makes them outliers? Compute their average spending per feature and compare to the cluster centroids. Are they extreme spenders or minimal spenders?

2. **K-Means++ vs random init.** Run `KMeans(n_clusters=5, init='random', n_init=1)` ten times with different seeds and record inertia. Then run `init='k-means++'`. Plot both inertia distributions as histograms. How much more stable is K-Means++?

3. **Segment radar chart.** Create a radar (spider) plot showing the centroid profile for each of the 5 segments. Which features most distinguish "Big Spenders" from "Price-Sensitive"?

In [ ]:
# Exercise 1 — DBSCAN noise analysis
# TODO: your solution here
pass

In [ ]:
# Exercise 2 — K-Means++ vs random init stability
# TODO: your solution here
pass

In [ ]:
# Exercise 3 — Segment radar chart
# TODO: your solution here
pass